# SAÉ — Migration vers NoSQL

## Contexte et contraintes métier

La base décrit la programmation d’une radio et ses diffusions. Les requêtes s’appuient notamment sur les règles suivantes :

- **Protection du jeune public** : aucun titre `explicit` ne doit être diffusé pendant les tranches protégées du matin et de l’après-midi.
- **Quota de nouveautés** : l’antenne doit diffuser un minimum de titres récent·e·s, sélectionné·e·s selon la date de sortie de leur album.
- **Déclaration des diffusions** : pour les sociétés de gestion de droits, il faut pouvoir obtenir rapidement, pour chaque artiste, le nombre de diffusions et la durée cumulée correspondante.

Pour chaque requête, choisissez un ordre de présentation lorsqu’il n’est pas évident, afin de pouvoir comparer les résultats après migration.

In [1]:
!pip install pandas --quiet

In [3]:
import sqlite3
import pandas as pd
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

# Connection to Radio Kiosque database
sqlite_connection = sqlite3.connect("data/RadioKiosque.sqlite")

Comment requêter la base de données ?

In [4]:
pd.read_sql_query(
    """
        SELECT host, title
        FROM Shows
        ORDER BY host
    """,
    con=sqlite_connection
)

,host,title
0,Anouk Lindqvist,Le Réveil Kiosque
1,Anouk Lindqvist,Rush Hour
2,DJ Marvelle,Les Insomniaques
3,Dario Fenn,Tour de Piste
4,Françoise Dubreuil,Dimanche Vinyle
5,Karim Benali,Café Croissant
6,Karim Benali,Nouveautés Kiosque
7,Orla Quinn,Pause Midi
8,Orla Quinn,Weekend Matin
9,Salomé Ndiaye,Scène Ouverte


In [5]:
#Première analyse pour découvrir la base de données

#Nombre de lignes dans quelques tables 

pd.read_sql_query(
    """
        SELECT 
        (SELECT COUNT(*) FROM Shows) AS "Nombre de Shows",
        (SELECT COUNT(*) FROM Broadcasts) AS "Nombre de broadcasts",
        (SELECT COUNT(*) FROM listeners) AS "Nombre d'auditeurs",
        (SELECT COUNT(*) FROM requests) AS "Nombre de requêtes",
        (SELECT COUNT(*) FROM Tracks) AS "Nombre de chansons"
    """,
    con=sqlite_connection
)

,Nombre de Shows,Nombre de broadcasts,Nombre d'auditeurs,Nombre de requêtes,Nombre de chansons
0,12,200107,600,1626,11335


In [13]:
#Afficher quelques lignes de chansons explicites

pd.read_sql_query(
    """
        SELECT name as "Titre", explicit
        FROM Tracks
        WHERE explicit = 1
        LIMIT 10
    """,
    con=sqlite_connection
)

,Titre,explicit
0,Mon amour,1
1,F4ck 4ever,1
2,Girlfriend (feat. Dâm-Funk),1
3,Full of life,1
4,Big eye,1
5,Je pardonne,1
6,Mon cœur tu es fou,1
7,L'antidote,1
8,Il faudrait,1
9,Bonjour docteur,1


### 1. Auditeur·ices sans demande

Lister les auditeur·ices n’ayant jamais effectué de demande musicale.

In [39]:
Q1 = pd.read_sql_query(
    """
        SELECT Listeners.pseudo AS "Pseudonyme"
        FROM Listeners
        LEFT JOIN Requests ON Listeners.listener_id = Requests.listener_id
        WHERE Requests.Listener_id IS NULL
        ORDER BY Listeners.listener_id
    """,
    con=sqlite_connection
)

Q1.to_csv("references/q1.csv", index = False)

Q1

,Pseudonyme
0,solar_raven63
1,red_seal79
2,lunar_llama20
3,pink_lynx87
4,red_owl90
...,...
174,big_owl10
175,secret_sparrow56
176,fast_sparrow44
177,calm_raven24


### 2. Bilan par animateur·ice

Pour chaque animateur·ice, indiquer le nombre d’émissions, le nombre de diffusions et la durée totale diffusée.

In [40]:
Q2 = pd.read_sql_query(
    """
        SELECT Shows.host as "Animateur(trice)", 
            COUNT( DISTINCT Shows.show_id) as "Nombre d'émissions", 
            COUNT(Broadcasts.broadcast_id) as "Nombre de diffusions",
            SUM(Tracks.duration_ms) as "Durée totale diffusée"
        FROM Shows
        LEFT JOIN Broadcasts ON Shows.show_id = Broadcasts.show_id
        LEFT JOIN Tracks ON Broadcasts.track_id = Tracks.track_id
        GROUP BY Shows.host
        ORDER BY Shows.host
    """,
    con=sqlite_connection
)

Q2.to_csv("references/q2.csv", index = False)

Q2

,Animateur(trice),Nombre d'émissions,Nombre de diffusions,Durée totale diffusée
0,Anouk Lindqvist,2,43894,10140191003
1,DJ Marvelle,1,33201,7538676031
2,Dario Fenn,1,21830,5080002640
3,Françoise Dubreuil,1,6466,1510922539
4,Karim Benali,2,28522,6629138076
5,Orla Quinn,2,34592,8107701472
6,Salomé Ndiaye,2,17788,4045055699
7,Yasmine Tazi,1,13814,3016880054


### 3. Bilan par tranche horaire

Pour chaque tranche horaire, indiquer le nombre de diffusions et le nombre de titres explicites diffusés.

In [41]:
Q3 = pd.read_sql_query(
    """
        SELECT Shows.time_slot as "Tranche horaire",
            COUNT(Broadcasts.broadcast_id) AS "Nombre de diffusions", 
            SUM(Tracks.explicit) AS "Nombre de titres explicites"
        FROM Shows
        LEFT JOIN Broadcasts ON Shows.show_id = Broadcasts.show_id
        LEFT JOIN Tracks ON Broadcasts.track_id = Tracks.track_id
        GROUP BY Shows.time_slot
        Order By Shows.time_slot
    """,
    con=sqlite_connection
)

Q3.to_csv("references/q3.csv", index = False)

Q3

,Tranche horaire,Nombre de diffusions,Nombre de titres explicites
0,afternoon,49981,502
1,evening,51766,20210
2,morning,56270,566
3,night,42090,11729


### 4. Bilan par titre

Pour chaque titre, indiquer le nombre de diffusions, l’audience cumulée et le nombre d’émissions différentes où il a été programmé.

In [ ]:
Q4 = pd.read_sql_query(
    """
        SELECT Shows.time_slot as "Tranche horaire",
            COUNT(Broadcasts.broadcast_id) AS "Nombre de diffusions", 
            SUM(Tracks.explicit) AS "Nombre de titres explicites"
        FROM Shows
        LEFT JOIN Broadcasts ON Shows.show_id = Broadcasts.show_id
        LEFT JOIN Tracks ON Broadcasts.track_id = Tracks.track_id
        GROUP BY Shows.time_slot
        Order By Shows.time_slot
    """,
    con=sqlite_connection
)

Q4.to_csv("references/q4.csv", index = False)

Q4

### 5. Bilan par artiste

Pour chaque artiste (principal·e ou invité·e), indiquer le nombre de titres au catalogue, le nombre de diffusions et la durée cumulée diffusée. Inclure les artistes jamais diffusé·es. Trier par ordre décroissant du nombre de diffusions.

In [12]:
pd.read_sql_query(
    """
        SELECT Listeners.pseudo AS "Pseudonyme"
        FROM Listeners
        LEFT JOIN Requests ON Listeners.listener_id = Requests.listener_id
        WHERE Requests.Listener_id IS NULL
        ORDER BY Listeners.listener_id
    """,
    con=sqlite_connection
)

,Pseudonyme
0,solar_raven63
1,red_seal79
2,lunar_llama20
3,pink_lynx87
4,red_owl90
...,...
174,big_owl10
175,secret_sparrow56
176,fast_sparrow44
177,calm_raven24


### 6. Diffusions par décennie et tranche horaire

Présenter le nombre de diffusions sous forme de tableau croisé entre la décennie de sortie de l’album et la tranche horaire.

Voici le format attendu :

| decade | morning | afternoon | evening | night |
|-------:|--------:|----------:|--------:|------:|
|   1960 |         |           |         |       |
|   1970 |         |           |         |       |
|   1980 |         |           |         |       |
|   1990 |         |           |         |       |
|   2000 |         |           |         |       |
|   2010 |         |           |         |       |
|   2020 |         |           |         |       |

&#128161; Exécuter la requête en SQL avec la décennie, la tranche horaire et le nombre de diffusion. Puis pivoter la table avec la fonction `pivot_table` de Pandas.

In [10]:
pd.read_sql_query(
    """
        SELECT Listeners.pseudo AS "Pseudonyme"
        FROM Listeners
        LEFT JOIN Requests ON Listeners.listener_id = Requests.listener_id
        WHERE Requests.Listener_id IS NULL
        ORDER BY Listeners.listener_id
    """,
    con=sqlite_connection
)

,Pseudonyme
0,solar_raven63
1,red_seal79
2,lunar_llama20
3,pink_lynx87
4,red_owl90
...,...
174,big_owl10
175,secret_sparrow56
176,fast_sparrow44
177,calm_raven24


### 7. Audience par décennie et tranche horaire

Présenter l’audience cumulée sous forme de tableau croisé entre la décennie de sortie de l’album et la tranche horaire.

In [16]:
Q7 = pd.read_sql_query(
    """
        SELECT
            CAST(substr(Albums.release_date, 1, 3) AS INTEGER) * 10 AS decade,
            SUM(CASE WHEN Shows.time_slot = 'morning'   THEN Broadcasts.listeners ELSE 0 END) AS morning,
            SUM(CASE WHEN Shows.time_slot = 'afternoon' THEN Broadcasts.listeners ELSE 0 END) AS afternoon,
            SUM(CASE WHEN Shows.time_slot = 'evening'   THEN Broadcasts.listeners ELSE 0 END) AS evening,
            SUM(CASE WHEN Shows.time_slot = 'night'     THEN Broadcasts.listeners ELSE 0 END) AS night
        FROM Broadcasts
        JOIN Tracks ON Broadcasts.track_id = Tracks.track_id
        JOIN Albums ON Tracks.album_id = Albums.album_id
        JOIN Shows ON Broadcasts.show_id = Shows.show_id
        GROUP BY decade
        ORDER BY decade
    """,
    con=sqlite_connection
)

#Q7.to_csv("references/q7.csv", index = False)

Q7

,decade,morning,afternoon,evening,night
0,1960,322812,205118,108856,28278
1,1970,401761,265248,142185,35318
2,1980,754338,484825,244668,65035
3,1990,1252052,826204,672995,151936
4,2000,2354505,1527890,815377,209362
5,2010,3853223,2500796,2190868,488681
6,2020,3628210,2332811,2507266,503677


### 8. Top 10 des audiences moyennes

Parmi les titres diffusés au moins trois fois, lister les dix titres ayant la plus forte audience moyenne par diffusion. Trier par l'audience moyenne.

In [8]:
Q8 = pd.read_sql_query(
    """
        SELECT Tracks.track_id, Tracks.name, COUNT(*) AS number_broadcasts, AVG(Broadcasts.listeners) AS average_audience
        FROM Broadcasts
        JOIN Tracks ON Broadcasts.track_id = Tracks.track_id
        GROUP BY Tracks.track_id, Tracks.name
        HAVING COUNT(*) >= 3
        ORDER BY average_audience DESC
        LIMIT 10
    """,
    con=sqlite_connection
)

Q8.to_csv("references/q8.csv", index = False)

Q8

,track_id,name,number_broadcasts,average_audience
0,7nRCeBY3Oyqu94Nav3ViTT,Miroirs,17,210.529412
1,4uIXvy8PbYLEhs0PZgaP8p,Delirium (feat. Mark E. Smith),13,209.769231
2,65fMfPqIzVV9XAzQ4UvkmN,Elle ecoute pousser les fleurs - Remastered,9,209.111111
3,7nzsY8vlnKdvGOEE0rjAXZ,Arabella,15,206.933333
4,2LKRpB6YTz0MUqbhJoqNCg,Back It Up,15,206.733333
5,2Hb8HUCnMZ74gwl4DppYDd,Sampha's Plea,797,206.424090
6,76HKOVWqsEg26SECtmw7Rz,The Way You Make Me Feel - 2012 Remaster,12,205.666667
7,40o1yxXRNZqbKLQOqFJKp0,RA TA TA,25,204.840000
8,6H3kDe7CGoWYBabAeVWGiD,Gimme Shelter,666,202.632132
9,720ALyzJaCOQRVxNMdjwCO,Dors,17,201.705882


### 9. Infractions à la protection du jeune public

Lister chaque diffusion d’un titre explicite dans une émission protégée, avec le titre, l’identifiant de l’émission et l’horodatage.

In [17]:
Q9 = pd.read_sql_query(
    """
        SELECT Tracks.name AS track_title, Broadcasts.show_id, Broadcasts.played_at
        FROM Broadcasts
        JOIN Tracks ON Broadcasts.track_id = Tracks.track_id
        JOIN Shows ON Broadcasts.show_id = Shows.show_id
        WHERE Tracks.explicit = 1 AND Shows.is_protected = 1
        ORDER BY Broadcasts.played_at
    """,
    con=sqlite_connection
)

#Q9.to_csv("references/q9.csv", index = False)

Q9

,track_title,show_id,played_at
0,GOD.,11,2023-10-08 11:46:02
1,Je réponds plus,1,2023-10-10 08:30:50
2,3 PALI,1,2023-10-11 07:14:02
3,Criminel,4,2023-10-12 16:49:48
4,Red Bottom Sky,4,2023-10-13 15:10:52
...,...,...,...
1063,DIEU MERCI RELOADED,2,2026-09-29 09:06:28
1064,Ma cousin,3,2026-09-29 12:57:55
1065,Yusuf,4,2026-09-29 15:26:47
1066,Trop beau - live acoustique,4,2026-09-30 16:12:11


### 10. Albums diffusés en entier

Lister les albums dont tous les titres ont été diffusés au moins une fois.

In [18]:
Q10 = pd.read_sql_query(
    """
        SELECT Albums.album_id, Albums.name
        FROM Albums
        JOIN Tracks ON Tracks.album_id = Albums.album_id
        LEFT JOIN Broadcasts ON Broadcasts.track_id = Tracks.track_id
        GROUP BY Albums.album_id, Albums.name
        HAVING COUNT(DISTINCT Tracks.track_id) = COUNT(DISTINCT Broadcasts.track_id)
        ORDER BY Albums.name
    """,
    con=sqlite_connection
)

#Q10.to_csv("references/q10.csv", index = False)

Q10

,album_id,name
0,7MwLBcJxdS3LjwX9y2H3jq,( ) [2022 Remaster]
1,77wWx9sOCJiy0wcn0P44NO,10:35
2,6h8MihL1baSyYITmOxrIxP,24 Hours
3,2v5hO6fszW8uKgA84BdxtY,4 de Fevereiro
4,3gmPezlcMPrSBWGH0jyk29,40K (Live)
...,...,...
215,6l4WefLj0rzKsHc2sNgkwG,na błysk
216,5X9dDbpm5lex928d1Ih3Ia,seule dans la foule
217,3T4tUhGYeRNVUGevb0wThu,÷ (Deluxe)
218,4XFjpLTwBHNkPFuJcE59me,Кароока


### 11. Podcasts classés par pays

Pour chaque pays, indiquer le nombre de podcasts classés, le nombre de podcasts explicites et le nombre de podcasts dont la fiche est fiable.

In [19]:
Q11 = pd.read_sql_query(
    """
        SELECT PodcastRankings.country, COUNT(*) AS nb_podcasts, SUM(Podcasts.explicit) AS nb_explicit, SUM(Podcasts.match_status = 'exact') AS nb_exact_match
        FROM PodcastRankings
        JOIN Podcasts ON PodcastRankings.podcast_id = Podcasts.podcast_id
        GROUP BY PodcastRankings.country
        ORDER BY PodcastRankings.country
    """,
    con=sqlite_connection
)

#Q11.to_csv("references/q11.csv", index = False)

Q11

,country,nb_podcasts,nb_explicit,nb_exact_match
0,AT,100,34,100
1,BE,100,30,100
2,CH,100,33,100
3,DE,100,37,100
4,DK,100,47,100
5,ES,100,29,100
6,FI,100,55,100
7,FR,101,29,101
8,GB,100,70,100
9,IE,100,63,100


### 12. Podcasts programmables

Lister les podcasts programmables, avec leur nom, leur éditeur, le nombre de pays dans le top 10 desquels ils figurent et leur meilleur rang. Trier du plus présent au moins présent, puis par meilleur rang.

In [20]:
Q12 = pd.read_sql_query(
    """
        SELECT Podcasts.podcast_id, Podcasts.name AS podcast_name, Podcasts.publisher, COUNT(*) AS nb_countries, MIN(PodcastRankings.rank) AS best_rank
        FROM Podcasts
        JOIN PodcastRankings ON PodcastRankings.podcast_id = Podcasts.podcast_id
        WHERE Podcasts.match_status = 'exact' AND Podcasts.explicit = 0 AND PodcastRankings.rank <= 10
        GROUP BY Podcasts.podcast_id, Podcasts.name, Podcasts.publisher
        ORDER BY nb_countries DESC, best_rank ASC
    """,
    con=sqlite_connection
)

#Q12.to_csv("references/q12.csv", index = False)

Q12

,podcast_id,podcast_name,publisher,nb_countries,best_rank
0,0xVTjXuo1jKKi2lyv0cRmI,Aha! Zehn Minuten Alltags-Wissen,WELT,4,1
1,4xQ0IUSsrfs6pltTnwr3Kk,LEGEND,Guillaume Pley,4,5
2,2HGcJRYrjGnpce6bRp8UXm,The Psychology of your 20s,iHeartPodcasts,3,1
3,4us7Lec3iNeAXDoG56Vths,5 Minuten Harry Podcast von Coldmirror,funk - von ARD und ZDF,3,5
4,2OyqGF8rokIi7K0CDwQc1Z,Ray William Johnson: True Story Podcast,Ray William Johnson,3,8
5,1Tn0rNAr68DU0gLZXWiptV,CHRONIQUES CRIMINELLES,Jacques Pradel - TF1,2,2
6,1HCF9OMp8ItCvFT1P3rC7x,Terra X History - Der Podcast,ZDF - Terra X,2,4
7,7iQXmUT7XGuZSzAMjoNWlX,The Diary Of A CEO with Steven Bartlett,FlightStory,2,6
8,16zDAJSEJp06lZnPdHv8pI,Die Deutschen Podcast,Nizar und Shayan,2,7
9,7AH8nT1CoTmV1MfhxyTl3n,SAFE PACE - Le podcast des sports d'endurance,"Hugo Clément, Charlotte Morel, Hugo Tormento e...",2,7


### 13. Durée et activité des podcasts programmables

Pour chaque podcast programmable, indiquer le nombre d’épisodes connus, la durée moyenne de ses épisodes (en minutes) et la date de son dernier épisode. Ne garder que les podcasts dont la durée moyenne ne dépasse pas une heure.

In [24]:
Q13 = pd.read_sql_query(
    """
        SELECT Podcasts.podcast_id, Podcasts.name AS podcast_name, COUNT(*) AS nb_episodes, ROUND(AVG(PodcastEpisodes.duration_ms) / 60000.0, 1) AS avg_minutes, MAX(PodcastEpisodes.release_date) AS last_release
        FROM Podcasts
        JOIN PodcastEpisodes ON PodcastEpisodes.podcast_id = Podcasts.podcast_id
        WHERE Podcasts.match_status = 'exact' AND Podcasts.explicit = 0
        GROUP BY Podcasts.podcast_id, Podcasts.name
        HAVING AVG(PodcastEpisodes.duration_ms) / 60000.0 <= 60
        ORDER BY Podcasts.podcast_id
    """,
    con=sqlite_connection
)

#Q13.to_csv("references/q13.csv", index = False)

Q13

,podcast_id,podcast_name,nb_episodes,avg_minutes,last_release
0,00VXbMpd9bHNxc4p08qDjl,Luxembourgish with Anne PODCAST,5,11.8,2026-10-15
1,01tskftKSz6fmBViCZfrls,Hoss & Hopf,5,41.8,2025-12-05
2,02RUOjUej5WoawxQWISAW9,Wissen Weekly,5,29.1,2026-10-11
3,033GExaONGcSNNvGj5ujUa,JYP Podcast,5,21.2,2026-10-12
4,033IzfPM2wYNjpMbG04UBT,NO GATEKEEPING með Ástrós Trausta,5,51.5,2026-08-23
...,...,...,...,...,...
507,7wBglag9syy3J5gHkRpKFh,Bleib entspannt! Der Familien-Podcast für magi...,5,43.8,2026-10-18
508,7wIDnmJ41exqZZ5GnsBGDS,Joel Osteen Podcast,5,27.3,2026-10-18
509,7wmClvYJGCKo7Pjs6AlIOg,Globally - La geopolitica spiegata in modo chiaro,5,23.1,2026-10-16
510,7yHXT4B4L7iWBNMDZS8y2Y,Nooit Geweten,5,19.6,2026-09-25


In [ ]:
# Close the connection to the database
sqlite_connection.close()